In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display


In [ ]:
# 1. Config
ROOT = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Final")
DATA = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Final/output/merged_all_11participants_8min.csv")  # same input as H1_to_H3.R / H4_multivariate.R
CORE_PHASES = ["BikeU", "BikeG", "WalkU", "WalkG"]  # x <= 0 counts use the analysed core phases only
PAPERS = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Github/Experiment papers")
OUT_DIR = ROOT / "scripts"
OUT_COUNTS = OUT_DIR / "Normalization_H1toH4_counts.csv"
OUT_INDEX = OUT_DIR / "Normalization_H1toH4_paper_index.csv"
OUT_PHYSIO = OUT_DIR / "Normalization_H1toH4_physio.csv"

# Wording of each transformation as modelled. HRV: ln(x), rows <= 0 excluded (Laborde, Mosley & Thayer 2017);
# never ln(1 + x). Must stay identical to H4_SPEC$outcome_transform in scripts/H4_multivariate.R.
STEP5_TEXT = {
    "No transformation": "No transformation",
    "Natural log": "Natural log, ln(x); rows <= 0 excluded",
}
# Physiological outcomes fitted in H4 Step 5 (H4_SPEC$outcomes in scripts/H4_multivariate.R).
# empatica__eda_scl_usiemens is not a Step 5 outcome.
NOT_IN_STEP5 = {"empatica__eda_scl_usiemens"}

# Parameter, the transformation, and how many papers used that transformation.
# One row per parameter. Counted only when the paper uses that transformation.
# A paper that only rejects another method is not counted.

PAPER_GROUP_1 = ["Fowles", "EDA_recommendations_BIOPAC", "Vocks et al 2021", "Zhang et al 2020", "Figner_Murphy_2010"]
PAPER_GROUP_2 = ["Laborde Mosley Thayer 2017", "An Overview of Heart Rate Variability Metrics and Norms", "HRV_task_force", "Reliability of Ultra-Short-Term", "Threshold-Based Artefact Correction"]
PAPER_GROUP_3 = ["HRV_task_force", "An Overview of Heart Rate Variability Metrics and Norms", "Laborde Mosley Thayer 2017", "Reliability of Ultra-Short-Term", "Threshold-Based Artefact Correction"]
PAPER_GROUP_4 = ["Imburgio et al 2024", "Beggiato et al 2018", "Mahanama et al 2021", "Wang et al 2025", "Wang et al 2024"]
PAPER_GROUP_5 = ["Wang et al 2025", "Wang et al 2024", "Evans et al 2012", "Tomasi et al 2016", "Rupi and Krizek 2019", "Trefzger et al 2018", "Vansteenkiste et al 2014"]
PAPER_GROUP_6 = ["Wang et al 2025", "Wang et al 2024", "Evans et al 2012", "Tomasi et al 2016", "Fleming et al 2024"]

USED = {
    "Heart rate": [("No transformation", ["Zhang et al 2020", "Beggiato et al 2018", "Outdoor temperature heart rate", "Personal_thermal_comfort_wearable_sensors_2018", "Thermal_comfort_physiological_sensing_DTU_2025", "Tervonen"])],
    "Skin temperature": [("No transformation", ["Zhang et al 2020", "Personal_thermal_comfort_wearable_sensors_2018", "Thermal_comfort_physiological_sensing_DTU_2025", "Kyriakou", "novel methodology for human thermal comfort"])],
    "Skin conductance level": [("No transformation", PAPER_GROUP_1)],
    "Skin conductance, tonic": [("No transformation", PAPER_GROUP_1)],
    "Skin conductance, phasic": [("No transformation", ["Fowles", "EDA_recommendations_BIOPAC", "Vocks et al 2021", "EDA_transform_BIOPAC_EDA_SCR", "Figner_Murphy_2010"])],
    "SDNN": [("Natural log", PAPER_GROUP_2)],
    "RMSSD": [("Natural log", PAPER_GROUP_2)],
    "LF power": [("Natural log", PAPER_GROUP_2)],
    "HF power": [("Natural log", PAPER_GROUP_2)],
    "VLF power": [("Natural log", PAPER_GROUP_2)],
    "Total power": [("Natural log", PAPER_GROUP_2)],
    "LF/HF ratio": [("Natural log", PAPER_GROUP_2)],
    "LF normalized": [("No transformation", PAPER_GROUP_3)],
    "HF normalized": [("No transformation", PAPER_GROUP_3)],
    "Pupil diameter": [("No transformation", PAPER_GROUP_4)],
    "Pupil change rate": [("No transformation", PAPER_GROUP_4)],
    "Eyelid aperture": [("No transformation", ["Mahanama et al 2021", "Beggiato et al 2018", "Wang et al 2025", "Wang et al 2024", "Evans et al 2012"])],
    "Blink duration": [("No transformation", PAPER_GROUP_5)],
    "Fixation duration": [("No transformation", PAPER_GROUP_5)],
    "Saccade duration": [("No transformation", PAPER_GROUP_5)],
    "Fixation rate": [("No transformation", PAPER_GROUP_5)],
    "Saccade amplitude": [("No transformation", PAPER_GROUP_5)],
    "Saccade peak velocity": [("No transformation", PAPER_GROUP_5)],
    "Gaze velocity": [("No transformation", PAPER_GROUP_5)],
    "Gaze dispersion": [("No transformation", PAPER_GROUP_5)],
    "Distance from center": [("No transformation", PAPER_GROUP_6)],
    "Gaze centrality": [("No transformation", PAPER_GROUP_6)],
}

COL = {
    "Heart rate": "empatica_heart_rate",
    "Skin temperature": "empatica_skin_temperature",
    "Skin conductance level": "empatica__eda_scl_usiemens",
    "Skin conductance, tonic": "empatica_eda_tonic",
    "Skin conductance, phasic": "empatica_eda_phasic",
    "SDNN": "empatica_hrv_td_sdnn",
    "RMSSD": "empatica_hrv_td_rmssd",
    "LF power": "empatica_hrv_fd_lf",
    "HF power": "empatica_hrv_fd_hf",
    "VLF power": "empatica_hrv_fd_vlf",
    "Total power": "empatica_hrv_fd_total_power",
    "LF/HF ratio": "empatica_hrv_fd_lf_hf_ratio",
    "LF normalized": "empatica_hrv_fd_lfnu",
    "HF normalized": "empatica_hrv_fd_hfnu",
    "Pupil diameter": "eyetracker_pupil_diameter_avg",
    "Pupil change rate": "eyetracker_pupil_change_rate",
    "Eyelid aperture": "eyetracker_eyelid_aperture_avg",
    "Blink duration": "eyetracker_blink_duration_s",
    "Fixation duration": "eyetracker_fixation_duration_s",
    "Saccade duration": "eyetracker_saccade_duration_s",
    "Fixation rate": "eyetracker_fixation_rate",
    "Saccade amplitude": "eyetracker_saccade_amplitude",
    "Saccade peak velocity": "eyetracker_saccade_peak_velocity",
    "Gaze velocity": "eyetracker_gaze_velocity",
    "Gaze dispersion": "eyetracker_gaze_dispersion",
    "Distance from center": "eyetracker_distance_from_center",
    "Gaze centrality": "eyetracker_gaze_centrality",
}

UNIT = {
    "empatica_heart_rate": "bpm", "empatica_skin_temperature": "°C", "empatica__eda_scl_usiemens": "microsiemens",
    "empatica_eda_tonic": "microsiemens", "empatica_eda_phasic": "microsiemens", "empatica_hrv_td_sdnn": "ms",
    "empatica_hrv_td_rmssd": "ms", "empatica_hrv_fd_lf": "ms2", "empatica_hrv_fd_hf": "ms2", "empatica_hrv_fd_vlf": "ms2",
    "empatica_hrv_fd_total_power": "ms2", "empatica_hrv_fd_lf_hf_ratio": "ratio", "empatica_hrv_fd_lfnu": "%",
    "empatica_hrv_fd_hfnu": "%", "eyetracker_pupil_diameter_avg": "mm", "eyetracker_pupil_change_rate": "mm/s",
    "eyetracker_eyelid_aperture_avg": "mm", "eyetracker_blink_duration_s": "s", "eyetracker_fixation_duration_s": "s",
    "eyetracker_saccade_duration_s": "s", "eyetracker_fixation_rate": "per second", "eyetracker_saccade_amplitude": "degrees",
    "eyetracker_saccade_peak_velocity": "degrees/s", "eyetracker_gaze_velocity": "degrees/s",
    "eyetracker_gaze_dispersion": "degrees", "eyetracker_distance_from_center": "pixels", "eyetracker_gaze_centrality": "proportion",
}
assert set(USED) == set(COL), set(USED) ^ set(COL)
assert set(COL.values()) == set(UNIT), set(COL.values()) ^ set(UNIT)
assert {t for opts in USED.values() for t, _ in opts} <= set(STEP5_TEXT)


In [ ]:
# 2. Load
raw = pd.read_csv(DATA, low_memory=False)
missing_cols = [c for c in COL.values() if c not in raw.columns]
assert not missing_cols, missing_cols

paper_files = tuple(p for p in PAPERS.rglob("*") if p.suffix.lower() == ".pdf")
paper_paths = {}
for key in dict.fromkeys(key for options in USED.values() for _, keys in options for key in keys):
    hits = [p for p in paper_files if key in p.name]
    if len(hits) != 1:
        raise FileNotFoundError(f"{key} -> {[str(p) for p in hits]}")
    paper_paths[key] = hits[0]
core_values = raw.loc[raw.PhaseID.isin(CORE_PHASES), list(COL.values())].apply(pd.to_numeric, errors="coerce")
nonpositive_counts = core_values.le(0).sum()


In [ ]:
# 3. Define
def step5_text(col_name, transformation):
    return "Not in Step 5" if col_name in NOT_IN_STEP5 else STEP5_TEXT[transformation]

def check_log_domain(col_name, transformation):
    return int(nonpositive_counts[col_name]) if transformation == "Natural log" else 0


In [ ]:
def make_normalization_tables():
    catalog = []
    number_of = {}
    rows = []
    physio_rows = []
    for parameter, options in USED.items():
        transformation, keys = options[0]
        marks, titles = [], []
        for key in keys:
            path = paper_paths[key]
            if path.name not in number_of:
                number_of[path.name] = len(catalog) + 1
                catalog.append({"Number": f"[{number_of[path.name]}]", "Paper title": path.name, "File path": str(path)})
            marks.append(f"[{number_of[path.name]}]")
            titles.append(path.name)
        col_name = COL[parameter]
        rows.append({
            "col_name": col_name,
            "Parameter": parameter,
            "Transformation": transformation,
            "Number of papers": len(marks),
            "Papers": ", ".join(marks),
        })
        physio_rows.append({
            "Column": col_name,
            "Parameter": parameter,
            "Unit": UNIT[col_name],
            "Transformation in the papers": transformation,
            "Transformation in Step 5": step5_text(col_name, transformation),
            "Rows excluded (x <= 0)": check_log_domain(col_name, transformation),
            "Paper titles": " | ".join(titles),
        })
    
    return pd.DataFrame(rows), pd.DataFrame(catalog), pd.DataFrame(physio_rows)


In [ ]:
count, papers_table, physio = make_normalization_tables()
assert not physio["Transformation in Step 5"].str.contains(r"1 \+ x|log1p", regex=True).any()
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 80)
display(count)
display(papers_table)
display(physio)


In [ ]:
# Save (single output cell)
count.to_csv(OUT_COUNTS, index=False)
papers_table.to_csv(OUT_INDEX, index=False)
physio.to_csv(OUT_PHYSIO, index=False)
print(OUT_COUNTS, OUT_INDEX, OUT_PHYSIO, sep="\n")
